# 06 · UI blocks — what was running when the interface stopped

A freeze that the user has to kill by hand used to produce **zero incidents**.
That was never evidence of health: every instrument for a stalled GUI ran on the
thread that stalls, so during the stall none of them fired and the log read
clean. A freeze measured from inside the freeze always returns zero.

`ui/block` is the probe that survives the condition it measures. The UI thread
only stamps a timestamp on a 50 ms timer; a plain OS thread — which the GUI
cannot block — decides whether that stamp has gone stale, measures the gap when
the thread returns, and files the incident.

This page asks the follow-up question: **which background work was running when
the interface stopped?** The chores are the suspects — the title/LLM chore, the
session scans, and the sidebar re-merge all run on a timer and all touch state
the render path reads.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) != "notebooks" else os.getcwd()))
sys.path.insert(0, os.path.abspath("."))
import ytrace_helpers as H

WINDOW = os.environ.get("YGG_NOTEBOOK_WINDOW", "30m")
HOST = H.GUI_HOST
H.describe_source(HOST, WINDOW)

In [ ]:
BLOCK_THRESHOLD_MS = 200      # ytrace::diagnosis::UI_BLOCK_THRESHOLD_MS
BLOCK_SEVERE_MS    = 1_000    # ytrace::diagnosis::UI_BLOCK_SEVERE_MS
BLOCK_DENSITY_WARN = 6.0      # ytrace::diagnosis::UI_BLOCK_DENSITY_PER_MIN

blocks = [r for r in H.tail(HOST, since=WINDOW, category="ui") if r.get("name") == "block"]
inc    = H.incidents(HOST, since=WINDOW)
ui_inc = [i for i in inc if "ui_block" in str(i.get("payload", {}).get("incident_id", ""))
          or "ui_block" in str(i.get("incident_id", ""))]

print(f"ui/block spans   : {len(blocks)}")
print(f"incidents total  : {len(inc)}   of which ui_block: {len(ui_inc)}")
if not blocks and not ui_inc:
    print("\nNo blocks recorded. That is only good news if the watchdog is RUNNING —")
    print("check that the GUI on this host is on a build that carries it, because")
    print("'no blocks' and 'no watchdog' are the same empty result set.")

In [ ]:
# Distribution of block durations. The tail is the freeze; the body is the
# jank that precedes one.
gaps = [r["duration_ms"] for r in blocks if isinstance(r.get("duration_ms"), (int, float))]
st = H.percentiles(gaps)
print("block duration (ms):", st)
if gaps:
    print("series:", H.sparkline(gaps))
    severe = [g for g in gaps if g >= BLOCK_SEVERE_MS]
    print(f"\nblocks >= {BLOCK_SEVERE_MS} ms (a freeze a person notices): {len(severe)} of {len(gaps)}")
    if severe:
        print("  worst:", ", ".join(f"{g/1000:.1f}s" for g in sorted(severe, reverse=True)[:8]))

In [ ]:
# ATTRIBUTION — what ran immediately before each gap opened.
attribution = H.Counter()
worst_by_activity = {}
for r in blocks:
    p = r.get("payload") or {}
    if not isinstance(p, dict):
        continue
    who = p.get("last_activity") or "unattributed"
    attribution[who] += 1
    gap = r.get("duration_ms") or 0
    if gap > worst_by_activity.get(who, 0):
        worst_by_activity[who] = gap

rows = [{"last_activity": k, "blocks": v, "worst_ms": worst_by_activity.get(k)}
        for k, v in attribution.most_common(20)]
print(H.table(rows, ["last_activity", "blocks", "worst_ms"]) if rows
      else "(no attributed blocks in this window)")
unattributed = attribution.get("unattributed", 0)
if unattributed:
    print(f"\n{unattributed} block(s) could not be attributed. An unattributed block is still a "
          "real block — the hint is best-effort by design and is skipped rather than "
          "waited for, so that recording one never costs the UI thread a lock.")

In [ ]:
# THE SUSPECTS — chore activity on the same bucket grid as the blocks, so a
# chore that lines up with the stalls is visible rather than inferred.
BUCKET_MS = 60_000

chores = {
    "title generation": [r for r in H.tail(HOST, since=WINDOW, category="copy_generation")
                         if r.get("name") == "title"],
    "background scan":  H.tail(HOST, since=WINDOW, category="background"),
    "sidebar merge":    [r for r in H.tail(HOST, since=WINDOW, category="sidebar")
                         if r.get("name") == "merge_rows"],
}

grids = {"ui blocks": H.bucket_by_time(blocks, BUCKET_MS)}
for label, rows_ in chores.items():
    grids[label] = H.bucket_by_time(rows_, BUCKET_MS)

all_buckets = sorted({b for g in grids.values() for b in g})
if all_buckets:
    lo, hi = all_buckets[0], all_buckets[-1]
    grid = list(range(lo, hi + BUCKET_MS, BUCKET_MS))
    width = max(len(k) for k in grids)
    for label, g in grids.items():
        counts = [len(g.get(b, [])) for b in grid]
        print(f"{label:{width}} {H.sparkline(counts)}  max={max(counts) if counts else 0}/min")
    print(f"{'':{width}} {H.ts(lo)} .. {H.ts(hi)}")
else:
    print("(nothing to align — no blocks and no chore activity in this window)")

In [ ]:
# Correlation, stated plainly rather than implied by two sparklines: in the
# minutes where blocks happened, was each chore busier than its own median?
if all_buckets:
    block_grid = grids["ui blocks"]
    blocky = [b for b in grid if block_grid.get(b)]
    print(f"{len(blocky)} of {len(grid)} minutes contained at least one block\n")
    for label, g in grids.items():
        if label == "ui blocks":
            continue
        allc = [len(g.get(b, [])) for b in grid]
        med = H.percentiles(allc).get("p50", 0)
        during = [len(g.get(b, [])) for b in blocky]
        if not during:
            print(f"{label:20} no blocky minutes to compare")
            continue
        dmed = H.percentiles(during).get("p50", 0)
        ratio = (dmed / med) if med else None
        verdict = ("no different" if ratio is None or 0.7 <= ratio <= 1.4
                   else ("BUSIER during blocks" if ratio > 1.4 else "quieter during blocks"))
        print(f"{label:20} median {med:.1f}/min overall, {dmed:.1f}/min during blocks"
              + (f" ({ratio:.2f}x) " if ratio else " ") + f"-> {verdict}")

In [ ]:
v = H.Verdict("UI thread blocks")

watchdog_live = bool(blocks) or bool(ui_inc)
if not watchdog_live:
    v.note(H.UNKNOWN, "watchdog reporting",
           "no ui/block records at all. 'no blocks' and 'no watchdog' produce the same "
           "empty result — confirm the GUI on this host runs a build carrying the "
           "watchdog before reading this as healthy.")
else:
    v.check("worst block", st.get("max"), f"< {BLOCK_SEVERE_MS} ms",
            warn_over=BLOCK_THRESHOLD_MS * 2, fail_over=BLOCK_SEVERE_MS, n=st.get("n", 0))
    v.check("block p95", st.get("p95"), f"< {BLOCK_SEVERE_MS} ms",
            warn_over=BLOCK_THRESHOLD_MS * 2, fail_over=BLOCK_SEVERE_MS, n=st.get("n", 0))
    span_min = ((max(r["ts_ms"] for r in blocks) - min(r["ts_ms"] for r in blocks)) / 60000
                if len(blocks) > 1 else None)
    density = (len(blocks) / span_min) if span_min else None
    v.check("blocks per minute", density, f"< {BLOCK_DENSITY_WARN}/min",
            warn_over=BLOCK_DENSITY_WARN / 2, fail_over=BLOCK_DENSITY_WARN)

    top = attribution.most_common(1)
    if top and top[0][0] != "unattributed":
        who, n = top[0]
        v.note(H.WARN, "most-blamed activity",
               f"{who} preceded {n} of {len(blocks)} blocks "
               f"(worst {worst_by_activity.get(who, 0):.0f} ms) — the first suspect to move off "
               "the UI thread")
    elif top:
        v.note(H.UNKNOWN, "attribution",
               "the most common hint is 'unattributed' — blocks are being caught but not "
               "explained; widen the activity hint before chasing a suspect")
v.show()